# 歌枠 自動解析（Colab版）

このノートブックは、曲データベースの「歌枠」を自動で解析し、
結果を GitHub（Milli-Orbis-portal の `data/karaoke-shazam.js`）に反映します。

**使い方**: 上のメニュー「ランタイム → すべてのセルを実行」を押すだけ。

- 1セッションで新しい歌枠から順に最大60本ほど解析します
- 解析済みは `done` マークが付き、重複解析されません
- 途中で切断された場合は、②のセルだけ再度実行してください

In [ ]:
!pip install -q yt-dlp shazamio 2>&1 | tail -1
!apt-get install -y -qq ffmpeg 2>&1 | tail -1
print("準備完了")

In [ ]:
import os, subprocess
if not os.path.exists("Milli-Orbis-portal"):
    subprocess.run(["git", "clone", "-q", "https://github.com/tsukikage-R8/Milli-Orbis-portal.git"], check=True)
os.chdir("Milli-Orbis-portal")
subprocess.run(["git", "pull", "-q", "--rebase", "--autostash"], check=True)
print("リポジトリ準備完了:", os.getcwd())

## ① 接続テスト（YouTubeに通るか確認）

出力の最後に `取得: ...` と表示されれば成功。
`Sign in to confirm` と表示された場合はこのノートブックでは解析できません。

In [ ]:
!python tools/identify-song.py "https://www.youtube.com/watch?v=jNQXAC9IVRw" --max-duration 10 2>&1 | tail -3

## ② 解析を実行（自動ループ）

1回の実行で5本ずつ、最大12回（約60本）解析します。
「残り 0 件」になれば全歌枠の解析が完了です。
途中で止まった場合はこのセルだけ再実行してください。

In [ ]:
import re
for i in range(12):
    print("===== 実行 %d/12 =====" % (i + 1))
    r = subprocess.run(["python", "tools/analyze-karaoke.py", "--max-videos", "5", "--max-minutes", "90"],
                       capture_output=True, text=True)
    last = (r.stderr or "").strip().splitlines()[-1] if (r.stderr or "").strip() else ""
    print(last)
    m = re.search(r"残り (\d+) 件", last)
    if m and int(m.group(1)) == 0:
        print("全歌枠の解析が完了しました")
        break

## ③ 結果をサイトに反映（2択）

**A. 自動プッシュ（おすすめ）**: GitHub の Personal Access Token を1回だけ用意
- iPhone/PCのブラウザで https://github.com/settings/personal-access-tokens/new を開く
- Repository access で `Milli-Orbis-portal` を選択 → Permissions で **Contents → Read and write** に変更
- 生成されたトークンをコピーして次のセルの入力欄に貼り付け

**B. 手動アップロード**: トークン不要。解析済みファイルが自動でダウンロードされるので、
GitHub の `data/` フォルダで「Add file → Upload files」からアップロード

In [ ]:
from getpass import getpass
PAT = getpass("GitHub PAT を貼り付け（不要ならEnterでスキップ）: ").strip()
if PAT:
    subprocess.run(["git", "remote", "set-url", "origin",
                    "https://x-access-token:%s@github.com/tsukikage-R8/Milli-Orbis-portal.git" % PAT], check=True)
    subprocess.run(["git", "config", "user.email", "bot@milliorbis.local"])
    subprocess.run(["git", "config", "user.name", "Milli Orbis Bot"])
    print("PAT 設定完了")

In [ ]:
import os
try:
    PAT
except NameError:
    PAT = ""
if PAT:
    subprocess.run(["git", "add", "data/karaoke-shazam.js"], check=True)
    subprocess.run(["git", "commit", "-q", "-m", "chore: update karaoke shazam results (colab)"], check=False)
    subprocess.run(["git", "push", "-q", "origin", "HEAD"], check=True)
    print("プッシュ完了！ サイトに反映されました")
else:
    from google.colab import files
    files.download("data/karaoke-shazam.js")
    print("ダウンロードした karaoke-shazam.js を GitHub の data/ フォルダにアップロードしてください")